# 05 — Does the shape of the template matter? BLS vs TLS

Notebook 03 found Kepler-8 b's period with **BLS** (Box Least Squares). BLS slides a
*rectangle* across every trial period and asks which one fits the dips best. It is
fast, it is what the Kepler mission pipeline used, and it works.

But a real transit is not a rectangle. A star is a disc that is **dimmer at its
edge than its centre** (limb darkening), so as a planet crosses, the amount of light
it blocks changes smoothly: ingress and egress are curved, and the floor of the
transit is rounded rather than flat.

**TLS** (Transit Least Squares) searches with that realistic shape instead of a box.
The claim in [Hippke & Heller (2019)](https://arxiv.org/abs/1901.02015) is that
matching the true shape buys you roughly **10% better detection efficiency for small
planets**, plus a sharper peak — which means a more precise period.

This notebook tests that claim on a planet we already know the answer for. Same
star, same detrended curve, two search methods, published values to check against.

> **A note on cost.** TLS is much slower than BLS — the search cell below takes
> roughly a minute. By default `skyplay` runs it on a few cores rather than all of
> them, so your laptop stays usable; pass `use_threads=` to change that.

In [ ]:
from skyplay import data, detrend, periods, plotting, vetting

plotting.use_style()

target = data.TARGETS['kepler-8']
print(target)
print(target.note)
print(f'published period : {target.period} d')
print(f'published Rp/Rs  : {target.rp_rs}')

## The data

`load_stitched` does what notebooks 01 and 03 did by hand — search MAST, download
four quarters, normalize each, concatenate, drop NaNs — and then **caches the
result** to `data/cache/`. The first run downloads; every run after that is instant.

In [ ]:
lc = data.load_stitched('kepler-8')       # quarters 1-4, Kepler long cadence
print(f'{len(lc)} cadences over {(lc.time.max() - lc.time.min()).value:.0f} days')
print(f'time format: {lc.time.format} ({lc.time.scale})')
lc.scatter(s=1);

## Detrending — with a filter that does not eat the transit

Notebook 03 used lightkurve's `flatten()`, a Savitzky-Golay filter. Here we use
**wotan's Tukey biweight**, which is *robust*: it down-weights outlying points
instead of being dragged toward them. Since in-transit points are outliers to the
trend, a robust filter preserves transit depth noticeably better.

The window must be comfortably wider than the transit — the rule of thumb is at
least ~3x the duration. Kepler-8 b's transit lasts about 3 hours, so 0.5 days is
a safe choice. Set it too narrow and the filter will absorb the very signal you
are looking for.

In [ ]:
flat, trend = detrend.biweight_flatten(lc, window_days=0.5)

ax = lc.scatter(s=1, label='stitched')
trend.plot(ax=ax, color=plotting.SERIES[1], label='biweight trend')
ax.set_title('The trend the filter removes');

It is worth being precise about what detrending did and did not do. It removed a
**correlated, slow** wander. It did almost nothing to the **white noise** — the
independent scatter from cadence to cadence — because that is not a trend, and no
filter can average it away without also averaging away your signal.

So the useful accounting is: how big was the trend we removed, how big is the
per-point noise we are stuck with, and how deep is the transit we are hunting?

In [ ]:
import numpy as np

# Successive-difference estimator: insensitive to any remaining slow structure.
white_noise = np.diff(flat.flux.value).std() / np.sqrt(2)

print(f'trend amplitude removed : {np.ptp(trend.flux.value) * 1e6:7.0f} ppm   correlated, so a filter can remove it')
print(f'per-point white noise   : {white_noise * 1e6:7.0f} ppm   irreducible per cadence')
print(f'transit depth to find   :    8300 ppm   deeper than the noise, so this one is easy')

In [ ]:
flat.scatter(s=1);

## Search 1 — BLS, the box

We search 1–10 days. Nothing here knows the answer.

In [ ]:
bls = periods.bls_search(flat, period_min=1, period_max=10, n_periods=20_000)
print(bls.summary())
print(bls.compare_to(target.period))

## Search 2 — TLS, the real transit shape

Same curve, same period range. TLS builds its own period grid, spaced so adjacent
trial periods stay within a fraction of a transit duration of each other — which is
why you hand it a *range* rather than a grid.

**This cell is the slow one (~1 minute).**

In [ ]:
tls = periods.tls_search(flat, period_min=1, period_max=10)
print(tls.summary())
print(tls.compare_to(target.period))

## Comparing the two

Note that these are plotted as **separate panels, not overlaid on twin axes**. BLS
power and TLS SDE are different quantities in different units; putting them on a
shared y-axis would invite a comparison that means nothing.

What *is* comparable across the panels is the **location** and the **sharpness** of
the peak. The black line marks the published period.

In [ ]:
fig = plotting.compare_spectra([bls, tls], published=target.period);

### Reading the detection statistics

The numbers are not interchangeable:

- **BLS power** is in arbitrary units — useful for ranking peaks *within one search*,
  not across searches or stars.
- **TLS SDE** (Signal Detection Efficiency) is the peak height measured in standard
  deviations of the spectrum itself. That makes it roughly comparable between stars,
  which is why a threshold like **SDE > 7–9** is a meaningful rule of thumb for
  "worth a second look."

In [ ]:
print(f'{"":6s} {"period (d)":>12s} {"error vs published":>20s} {"Rp/Rs":>8s} {"statistic":>22s}')
for r in (bls, tls):
    err = abs(r.period - target.period) / target.period * 100
    print(f'{r.method:6s} {r.period:12.5f} {err:19.4f}% {r.rp_rs:8.4f} '
          f'{r.power_label + " = " + format(r.power, ".1f"):>22s}')

print(f'\npublished: period {target.period} d, Rp/Rs {target.rp_rs}')

## The fold, using the TLS ephemeris

Two views of the same fold. The full orbit is where you check for a **secondary
eclipse** at phase 0.5 — starlight blocked on the far side of the orbit, the
signature of a companion star rather than a planet. The zoom is where the transit
shape itself becomes visible: notice the curved ingress and the rounded floor.
That curvature is exactly what TLS's template models and BLS's box does not.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plotting.plot_folded(flat.time.value, flat.flux.value, tls.period, tls.epoch,
                     ax=axes[0], title='Full orbit — is there a secondary eclipse?')
plotting.plot_folded(flat.time.value, flat.flux.value, tls.period, tls.epoch,
                     ax=axes[1], phase_window=0.05, title='Zoomed — the transit shape');

## Vetting the candidate

Finding a period is the easy part. Notebook 04 covered why most transit-like signals
are impostors. `skyplay.vetting.vet` runs those checks and reports a verdict per
check — and passing them all means "still a candidate," never "a planet."

In [ ]:
report = vetting.vet(flat.time.value, flat.flux.value, tls.period, tls.epoch, halfwidth=0.02)
print(report.summary())

## Takeaways

1. **Both methods find the planet.** On a signal this strong — a hot Jupiter with a
   ~8,000 ppm transit — BLS is entirely sufficient. TLS's advantage is not in finding
   obvious things.

2. **TLS gives a sharper peak and a slightly more accurate period.** Compare the
   panels: the TLS peak is dramatically narrower. That precision matters when you
   need an ephemeris good enough to predict a future transit, or to stack years of
   data without smearing.

3. **TLS costs real time.** Search wide with BLS while exploring; confirm with TLS
   once a candidate matters. That is the practical workflow.

4. **Neither method vets anything.** Both will happily hand you an eclipsing binary
   with a confident period and a beautiful peak. The statistic tells you a periodic
   dip exists; it says nothing about what caused it.

### Where TLS actually wins

This notebook is a *demonstration*, not a fair test of the 10% claim — Kepler-8 b is
far too easy. The honest experiment is an **injection-recovery test**: inject
synthetic transits of known small depth into a real light curve, at many periods and
depths, then measure what fraction each method recovers. `skyplay.synthetic` and
`skyplay.models.transit_model` have the pieces you need to build one, and it is a
genuinely good next project.